## Popularity recommendation
В этом методы мы будем предлагать пользователям самые высокооцененные фильмы и сериалы. То есть, мы отказываемся от персональных рекоммендаций, каждый пользователь будет получать в рекоммендациях одни и те же предложения.

Этот метод мы будем использовать в качестве baseline решения, далее все результаты мы будем сравнивать с результатами этого метода.

In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.data_loader import DataLoader
from src.utils import train_test_split_by_user
import pandas as pd

In [2]:
loader = DataLoader(path='../data')
ratings, movies, users = loader.load_all()

In [3]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,1193,5,2000-12-31 22:12:40
1,1,661,3,2000-12-31 22:35:09
2,1,914,3,2000-12-31 22:32:48
3,1,3408,4,2000-12-31 22:04:35
4,1,2355,5,2001-01-06 23:38:11


In [4]:
train, test = train_test_split_by_user(ratings)
train.head()

,userId,movieId,rating,timestamp
0,1,1193,5,2000-12-31 22:12:40
1,1,661,3,2000-12-31 22:35:09
2,1,914,3,2000-12-31 22:32:48
5,1,1197,3,2000-12-31 22:37:48
6,1,1287,5,2000-12-31 22:33:59


In [5]:
movie_stats = (
    train
    .groupby("movieId")
    .agg({
        "rating": ["mean", "count"]
    })
)

movie_stats.columns = ["mean_rating", "rating_count"]
movie_stats = movie_stats.reset_index()

In [6]:
movie_stats.head()

,movieId,mean_rating,rating_count
0,1,4.150888,1690
1,2,3.223958,576
2,3,3.065104,384
3,4,2.809160,131
4,5,3.052863,227


In [7]:
min_count = 50
popular_movies = movie_stats[movie_stats["rating_count"] >= min_count]
popular_movies.head()

,movieId,mean_rating,rating_count
0,1,4.150888,1690
1,2,3.223958,576
2,3,3.065104,384
3,4,2.809160,131
4,5,3.052863,227


In [8]:
popular_movies = popular_movies.sort_values(by="mean_rating", ascending=False)
popular_movies.head()

,movieId,mean_rating,rating_count
2675,2905,4.573770,61
1821,2019,4.559596,495
307,318,4.554305,1777
697,745,4.530303,528
49,50,4.524339,1438


In [9]:
popular_movies = popular_movies.merge(movies.drop(columns="genres", axis=1), on="movieId")
popular_movies.head()

,movieId,mean_rating,rating_count,title
0,2905,4.573770,61,Sanjuro (1962)
1,2019,4.559596,495,Seven Samurai (The Magnificent Seven) (Shichin...
2,318,4.554305,1777,"Shawshank Redemption, The (1994)"
3,745,4.530303,528,"Close Shave, A (1995)"
4,50,4.524339,1438,"Usual Suspects, The (1995)"


In [10]:
def get_recommendations(k=10): 
    return popular_movies["movieId"].head(k).tolist()

In [11]:
from src.metrics import Metrics

metrics = Metrics()

recommended = get_recommendations(10)

total_prec = 0
total_rec = 0
total_ndcg = 0

for user_id in users["userId"].unique():

    relevant = test[
        (test.userId == user_id) &
        (test.rating >= 4)
    ]["movieId"].tolist()

    precision, recall, ndcg = metrics.get_all(
        recommended,
        relevant,
        10
    )

    total_prec += precision
    total_rec += recall
    total_ndcg += ndcg

uniq_users = users['userId'].unique().shape[0]

print(f"total precision: {total_prec / uniq_users}\ntotal recall: {total_rec / uniq_users}\ntotal ndcg: {total_ndcg / uniq_users}")

total precision: 0.035827814569535425
total recall: 0.024897981142211503
total ndcg: 0.03358169538132602
